# 03. Serving the fine-tuned model with Ray Serve

**Part 3 of 3**

**Series:** [01 Ray and offline inference](01_ray_and_offline_inference.ipynb) · [02 Train and tune](02_train_and_tune.ipynb) · [03 Serve](03_serve.ipynb)

Notebook 02 produced a fine-tuned checkpoint. This notebook puts it behind an endpoint a chemist can use:
the 3D results viewer, prediction from Python, and swapping checkpoints without breaking what is already
in use.

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 2 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |

Section numbers follow the series as a whole, so this notebook starts at Section 6.


## 0. Setup

- The Workspace runs the `aiddd-boltz2-of3` image: Ray, Boltz-2 and 2 OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU-only head node. L4 GPU nodes join when a cell requests a GPU and leave when idle.
- Data, model weights and all benchmark results live on shared storage.

In [ ]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_LEGACY,
                            Boltz2Predictor, OF3Predictor, load_ligands, validate, make_scorer)
import notebook_helpers as wh

# True: run every GPU step live on GPUs.
# False: GPU steps load stored results. Serving is notebook 03.
USE_GPUS = False

# Data and weights are downloaded once (README, Setup). Weights are only needed with USE_GPUS = True.
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS]
if USE_GPUS:
    needed += [BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."
SERVICE_URL = os.environ.get("AIDDD_SERVICE_URL", "")   # the deployed prediction service used in notebook 03
GPU = "L4"            # GPU type for every GPU step in this notebook

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

In [ ]:
# Context from notebooks 01 and 02, loaded here so this notebook stands alone
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
ev71 = results["ev71-2a"]
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]     # the late follow-on used in 6.3
print("loaded benchmark results:", ", ".join(results))


## 6. Serving the fine-tuned model with Ray Serve

**2 parts:**

1. **6.1 to 6.3** The results viewer and live prediction.
2. **6.4** Swapping checkpoints and rolling out a new model safely.

`serve_viewer.py` is 1 Ray Serve application built from several deployments, each with its own resources
and scaling:

| Deployment | Runs on | Does |
|---|---|---|
| `Viewer` | 1 CPU | Web page: every crystal complex, a 3D view of each prediction, a SMILES box |
| Pose model | ½ L4 per replica | Predicts the pose of a new molecule with OpenFold3 |
| `Boltz2Live` | ½ L4 per replica | Predicts binding affinity with Boltz-2 |

**What to notice**

- **Pre- and post-processing are ordinary Python.** SMILES validation runs before the GPU models; pose
  rendering and the plain-language explanation run after. Each lives in the deployment that needs it.
- **Composition.** `Viewer` holds a handle to each model and calls both at once, so a request takes as long as
  the slower model, not the sum.
- **GPUs from every zone.** Model replicas scale between 1 and 4 and start in any availability zone with
  capacity (`enable_cross_zone_scaling` in `jobs/service.yaml`).
- **Same code, 2 settings:** `serve.run` in this Workspace, an Anyscale Service in production.

<img src="https://raw.githubusercontent.com/anyscale/templates/main/templates/cofolding-finetune-serve/docs/images/ray/serve-model-composition.png" alt="Composing pre-processing, models and post-processing" width="820">
<img src="https://raw.githubusercontent.com/anyscale/templates/main/templates/cofolding-finetune-serve/docs/images/ray/serve-autoscaling.svg" alt="Ray Serve autoscaling" width="480">
<sub>Diagrams: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 6.1 Open the viewer

Section 6 uses the prediction service already deployed as an Anyscale Service, so no GPU node has to start
here. The cell below shows both settings: the deployed service, and `serve.run` for the same application
inside a Workspace.


In [ ]:
from ray import serve
import serve_viewer

if SERVICE_URL:
    viewer_url = api_url = SERVICE_URL.rstrip("/")
    print(f"Deployed service: {viewer_url}")
else:
    # The same application, started in this Workspace on port 8000 instead of the deployed service
    assert USE_GPUS, "Set AIDDD_SERVICE_URL in the environment, or USE_GPUS = True to start the application here."
    serve.run(serve_viewer.build_app(accelerator=GPU), name="aiddd", route_prefix="/", blocking=False)
    viewer_url, api_url = wh.serve_link(), "http://localhost:8000"


In the viewer, choose a model tab and click a complex: the predicted compound (green) appears on the crystal
structure (orange).

### 6.2 Compounds for the live service

Paste any of these into the viewer's SMILES box, or into `MY_SMILES` below. All are EV-A71 2A late
follow-ons with a crystal structure, so you can compare the prediction with the experiment.

| Compound | Why it is interesting | SMILES |
|---|---|---|
| **x7101a** | 145k misses by 6.0 Å; fine-tuned models land within 0.4 Å | `CSc1ncnc(N2CCC3(CCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7101a** | Same series with a larger ring; 145k misses by 6.4 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7410a** | Potent (pKD 6.8), a different series; 145k misses by 5.6 Å | `CSc1ccc(-c2cnn[nH]2)cc1OC(=O)N1C[C@@H](C(N)=O)C[C@@H]2C[C@@H]21` |
| **x6714a** | Smaller scaffold; 145k misses by 4.6 Å | `NC(=O)[C@H]1CCC(=O)N(c2cc3cnccc3c(F)c2F)C1` |
| **x7259a** | Most potent shown here (pKD 7.9); Boltz-2 misses by 13 Å | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7259a, S → O** | A new molecule with no crystal structure | `COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

- A new molecule takes 1–3 minutes; a repeated molecule returns instantly from the cache.
- On the shared service the whole room shares the same GPUs, so requests may queue.

### 6.3 Prediction from Python

`predict` sends 1 SMILES string to the service. The optional `model` argument selects a checkpoint when the
service offers more than 1 (Section 6.4); without it the service uses its default model.

In [ ]:
import requests

def predict(smiles, model=None):
    t0 = time.time()
    headers = {"serve_multiplexed_model_id": model} if model else {}
    body = {"smiles": smiles, **({"model": model} if model else {})}
    r = requests.post(f"{api_url}/predict", json=body, headers=headers, timeout=900)
    out = r.json()
    if r.status_code != 200:
        raise RuntimeError(out.get("error", out))
    m = out.get("metrics", {})
    print(f"{out.get('status', 'done')} in {time.time() - t0:.0f} s · pose: {out.get('pose_model', model or 'default')}"
          f" · ipTM {out.get('pose_iptm') or float('nan'):.2f}"
          f" · P(binder) {m.get('affinity_probability_binary', float('nan')):.2f} (Boltz-2)")
    if out.get("cif"):
        out["cif_path"] = wh.MINE / "live" / f"{out.get('ligand_id', 'pred')}-{model or 'default'}.cif"
        out["cif_path"].parent.mkdir(parents=True, exist_ok=True)
        out["cif_path"].write_text(out["cif"])
    return out

NEW = "COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7259a with S replaced by O
out = predict(NEW)
if out.get("cif_path"):
    wh.show_analog(out["cif_path"], x7259a, label=f"new molecule ({out.get('pose_model', 'default')}, live)")

Grey ribbon and dark sticks: the crystal structure of x7259a. Green: the prediction for the new molecule,
superposed on that crystal. If the new molecule keeps its parent's binding mode, the 2 overlap.

Replace `MY_SMILES` with any compound from the table above, or a molecule of your own (up to 300 characters).

In [ ]:
MY_SMILES = "CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7101a
out = predict(MY_SMILES)
if out.get("cif"):
    wh.show_prediction(out["cif"], label=f"{MY_SMILES[:40]}… ({out.get('pose_model', 'default')})")

### 6.4 Swapping checkpoints and rolling out a new model

The fine-tuned model is ready. How does it reach chemists without breaking what they use today?

**Swap checkpoints on 1 deployment.** Ray Serve **model multiplexing** serves several checkpoints from the same
replicas and loads each on demand. A request names its checkpoint; the replica keeps the most recent ones in memory.

```python
@serve.deployment(ray_actor_options={"num_gpus": 0.5})
class PoseModel:
    @serve.multiplexed(max_num_models_per_replica=3)
    async def get_model(self, model_id: str):
        return load_checkpoint(CHECKPOINTS[model_id])     # 145k, ours, OpenBind release

    async def __call__(self, smiles: str):
        model = await self.get_model(serve.get_multiplexed_model_id())
        return model.predict(smiles)
```

The sequence below sends the same compound, x7101a, to each of the 3 checkpoints in turn. The predicted pose
moves from 6.0 Å off the crystal structure to within 0.5 Å.

In [ ]:
# Checkpoint names offered by the service; confirm these against your deployment
CHECKPOINTS = ["of3p2-145k", "ours", "of3p2-ft-ev2a"]
SWAP_DEMO = False     # True once the service offers multiple checkpoints

if SWAP_DEMO:
    for ck in CHECKPOINTS:
        out = predict(MY_SMILES, model=ck)
        if out.get("cif_path"):
            wh.show_analog(out["cif_path"], results["ev71-2a"].query("complex_name == 'A71EV2A-x7101a'").iloc[0],
                           label=f"x7101a predicted by {ck}", height=340)
else:
    print("Set SWAP_DEMO = True when the service offers several checkpoints.")

**Roll out safely with a canary.** An Anyscale Service can run the new version next to the current version and send
it a share of traffic first (for example 10%). Watch latency and errors, then promote it to 100% or roll back.

```bash
anyscale service deploy -f jobs/service.yaml --canary-percent 10     # new version takes 10% of traffic
anyscale service deploy -f jobs/service.yaml --canary-percent 100    # promote
anyscale service rollback -n aiddd-viewer                       # or roll back
```

| What Ray Serve and Anyscale add | Why it matters |
|---|---|
| **GPUs from every zone** | When 1 zone runs out of L4s, replicas start in another |
| **Composition in Python** | Pre-processing, models and post-processing each scale on their own |
| **Model multiplexing** | Several checkpoints on the same GPUs, chosen per request |
| **Canary rollouts and rollback** | A new model reaches a share of users first |

## 7. Summary and next steps

**Findings**

- **Generalization.** Models that reach 88 to 92% on SARS-CoV-2 late follow-ons fall to 1 to 24% on EV-A71 2A.
- **Fine-tuning.** 66 fragment-screen structures measurably improve OpenFold3 on held-out follow-on
  compounds (notebook 02, Section 5.6), and the resulting model is a specialist for that target.

**Delivered on a single Ray cluster**

- **Ray Data.** The full benchmark across 3 campaigns, and 402 new OpenFold3 predictions in 19 minutes on
  12 L4 GPUs at 80% GPU utilization.
- **Ray Train.** OpenFold3's own training code on 8 GPUs, with checkpoints every 100 steps and automatic
  recovery from a lost node.
- **Ray Serve.** Pose and affinity behind a single endpoint, with checkpoint swaps and canary rollouts.

The [README](README.md) contains the scientific background and the complete runbook.

Run the next cell to release the GPU nodes. The deployed service stays up; terminate it with
`anyscale service terminate -n aiddd-viewer` after the session.

In [ ]:
if SERVICE_URL:
    print("Nothing to stop: Section 6 used the deployed service, which stays up for the room.")
elif USE_GPUS:
    serve.shutdown()
    print("Serve application stopped. Idle GPU nodes will be released within a few minutes.")
